In [1]:
!nvidia-smi

Tue Apr 25 07:43:42 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   64C    P8    10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [15]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_candidates = [
    Path.cwd(),
    Path.cwd() / "Locality-iN-Locality-main",
    Path.cwd() / "Locality-iN-Locality",
]
repository_root = next(
    (path for path in repository_candidates if (path / "LNL_MoEx.py").exists()),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(repository_root)],
        check=True,
    )

os.chdir(repository_root)
print("Working directory:", Path.cwd())


Cloning into 'Locality-iN-Locality'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 36 (delta 8), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (36/36), 40.40 KiB | 3.37 MiB/s, done.


In [16]:
# Repository setup is handled in the previous cell for both Colab and local runs.
print("Repository ready:", Path.cwd())


/content/Locality-iN-Locality


In [ ]:
!pip install -q timm==0.9.16 einops scikit-learn


In [ ]:
import os
import math
import time
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
import timm

SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


In [ ]:
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)


## GTSRB

In [26]:
from urllib.request import urlretrieve

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
GTSRB_URLS = {
    "GTSRB_Final_Training_Images.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip",
    "GTSRB_Final_Test_Images.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip",
    "GTSRB_Final_Test_GT.zip": "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip",
}

for filename, url in GTSRB_URLS.items():
    destination = DATA_DIR / filename
    if not destination.exists():
        print("Downloading", filename)
        urlretrieve(url, destination)
    else:
        print("Already downloaded", filename)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  263M  100  263M    0     0  11.6M      0  0:00:22  0:00:22 --:--:-- 13.2M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84.8M  100 84.8M    0     0  9287k      0  0:00:09  0:00:09 --:--:-- 12.5M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99620  100 99620    0     0  46726      0  0:00:02  0:00:02 --:--:-- 46726


In [27]:
import zipfile

for filename in GTSRB_URLS:
    archive_path = DATA_DIR / filename
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

print("GTSRB archives extracted.")


Archive:  data/GTSRB_Final_Test_GT.zip
  inflating: data/GT-final_test.csv  


In [28]:
import shutil

In [29]:
data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')

test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)



with open('./data/GT-final_test.csv') as f:
  image_names = f.readlines()

for text in image_names[1:]:
  classes = int(text.split(';')[-1])
  image_name = text.split(';')[0]
  

  test_class_dir = os.path.join(test_dir, f"{classes:04d}")
  os.makedirs(test_class_dir, exist_ok=True)
  image_path = os.path.join(images_dir, image_name)

  shutil.copy(image_path, test_class_dir)

In [ ]:
class ResizeWithPadding:
    """Resize while preserving aspect ratio, then pad to a square image."""

    def __init__(self, size: int = 224, fill: int = 128):
        self.size = size
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        width, height = image.size
        scale = self.size / max(width, height)

        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )

        horizontal_padding = self.size - new_width
        vertical_padding = self.size - new_height

        left = horizontal_padding // 2
        right = horizontal_padding - left
        top = vertical_padding // 2
        bottom = vertical_padding - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=self.fill,
            padding_mode="constant",
        )


NORMALIZE_MEAN = (0.5, 0.5, 0.5)
NORMALIZE_STD = (0.5, 0.5, 0.5)

train_transform = transforms.Compose([
    ResizeWithPadding(size=224, fill=128),

    transforms.RandomApply([
        transforms.RandomAffine(
            degrees=10,
            translate=(0.05, 0.05),
            scale=(0.90, 1.10),
            shear=3,
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=128,
        )
    ], p=0.70),

    transforms.RandomPerspective(
        distortion_scale=0.15,
        p=0.20,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=128,
    ),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.02,
        )
    ], p=0.60),

    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

eval_transform = transforms.Compose([
    ResizeWithPadding(size=224, fill=128),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])


In [ ]:
TRAIN_ROOT = "./data/GTSRB/Final_Training/Images"
TEST_ROOT = "./data/GTSRB/test"

MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 2

index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
all_indices = np.arange(len(index_dataset))
all_targets = np.asarray(index_dataset.targets)

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=4000,
    random_state=SEED,
    shuffle=True,
    stratify=all_targets,
)

training_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=train_transform,
)
validation_dataset = torchvision.datasets.ImageFolder(
    TRAIN_ROOT,
    transform=eval_transform,
)
testset = torchvision.datasets.ImageFolder(
    TEST_ROOT,
    transform=eval_transform,
)

trainset = Subset(training_dataset, train_indices)
validationset = Subset(validation_dataset, validation_indices)

train_loader = DataLoader(
    trainset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    drop_last=True,
)

validation_loader = DataLoader(
    validationset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

test_loader = DataLoader(
    testset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

NUM_CLASSES = len(index_dataset.classes)

print("Training images:", len(trainset))
print("Validation images:", len(validationset))
print("Test images:", len(testset))
print("Classes:", NUM_CLASSES)


In [ ]:
# Visualization is intentionally omitted; it does not affect Top-1 accuracy.


In [ ]:
# Visualization is intentionally omitted; it does not affect Top-1 accuracy.


In [ ]:
# Visualization is intentionally omitted; it does not affect Top-1 accuracy.


## Final model: LNL-MoEx-S for GTSRB Top-1 accuracy


In [ ]:
# timm is installed in cell 3.


In [ ]:
# einops is installed in cell 3.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


## Baseline LNL-Ti disabled


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


## Baseline test disabled


In [ ]:
# Disabled: the final submission uses LNL-MoEx-S only.


## FGSM disabled — only clean Top-1 accuracy is required


In [ ]:
# Disabled.


## PGD disabled — only clean Top-1 accuracy is required


In [ ]:
# Disabled.


## Train the improved LNL-MoEx-S


In [ ]:
from LNL_MoEx import LNL_MoEx_S

model = LNL_MoEx_S(
    pretrained=False,
    num_classes=NUM_CLASSES,
    drop_path_rate=0.10,
)

# TNT-S ImageNet checkpoint referenced by the original repository.
TNT_S_PRETRAINED_URL = (
    "https://github.com/contrastive/pytorch-image-models/releases/"
    "download/TNT/tnt_s_patch16_224.pth.tar"
)

downloaded_checkpoint = torch.hub.load_state_dict_from_url(
    TNT_S_PRETRAINED_URL,
    map_location="cpu",
    check_hash=False,
)

if isinstance(downloaded_checkpoint, dict) and "state_dict" in downloaded_checkpoint:
    source_state = downloaded_checkpoint["state_dict"]
elif isinstance(downloaded_checkpoint, dict) and "model" in downloaded_checkpoint:
    source_state = downloaded_checkpoint["model"]
else:
    source_state = downloaded_checkpoint

clean_source_state = {}
for key, value in source_state.items():
    clean_key = key
    for prefix in ("module.", "model."):
        if clean_key.startswith(prefix):
            clean_key = clean_key[len(prefix):]
    clean_source_state[clean_key] = value

target_state = model.state_dict()

compatible_state = {
    key: value
    for key, value in clean_source_state.items()
    if key in target_state
    and value.shape == target_state[key].shape
    and not key.startswith("head.")
}

if not compatible_state:
    raise RuntimeError("No compatible TNT-S pretrained weights were found.")

model.load_state_dict(compatible_state, strict=False)
pretrained_parameter_names = set(compatible_state.keys())

del downloaded_checkpoint
del source_state
del clean_source_state


In [ ]:
model = model.to(device)


In [ ]:
@torch.inference_mode()
def calculate_top1(model: nn.Module, loader: DataLoader) -> float:
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = model(images)

        predictions = logits.argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / total


In [ ]:
NUM_EPOCHS = 100
MIN_EPOCHS = 40
EARLY_STOP_PATIENCE = 15
EARLY_STOP_MIN_DELTA = 0.02
TARGET_VALIDATION_TOP1 = 99.5
TARGET_STABLE_EPOCHS = 3
WARMUP_EPOCHS = 5
ACCUMULATION_STEPS = 4

BACKBONE_LR = 1e-4
NEW_LAYER_LR = 5e-4
MIN_LR_FACTOR = 0.01
WEIGHT_DECAY = 0.05

MOEX_LAMBDA = 0.90
MOEX_PROBABILITY = 0.50

# Prefer an explicit directory; otherwise use persistent Colab Drive when mounted
# and a local relative directory everywhere else.
checkpoint_dir_override = os.environ.get("CHECKPOINT_DIR")
if checkpoint_dir_override:
    CHECKPOINT_DIR = Path(checkpoint_dir_override).expanduser()
else:
    colab_drive_dir = Path("/content/drive/MyDrive")
    CHECKPOINT_DIR = (
        colab_drive_dir / "LNL_MoEx_checkpoints"
        if colab_drive_dir.exists()
        else Path.cwd() / "checkpoints"
    )
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "LNL_MoEx_S_GTSRB_top1.pth"
RESUME_TRAINING = os.environ.get("RESUME_TRAINING", "1").strip().lower() not in {"0", "false", "no", "off"}


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.10)

parameter_groups = {}

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_pretrained = name in pretrained_parameter_names
    learning_rate = BACKBONE_LR if is_pretrained else NEW_LAYER_LR

    no_weight_decay = (
        parameter.ndim == 1
        or name.endswith(".bias")
        or name in {"cls_token", "patch_pos", "pixel_pos"}
    )
    weight_decay = 0.0 if no_weight_decay else WEIGHT_DECAY

    group_key = (learning_rate, weight_decay)
    parameter_groups.setdefault(group_key, []).append(parameter)

optimizer = optim.AdamW(
    [
        {
            "params": parameters,
            "lr": learning_rate,
            "weight_decay": weight_decay,
        }
        for (learning_rate, weight_decay), parameters
        in parameter_groups.items()
    ],
    betas=(0.9, 0.999),
)

def learning_rate_factor(epoch: int) -> float:
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)

    progress = (
        epoch - WARMUP_EPOCHS
    ) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=learning_rate_factor,
)

scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


In [ ]:
def _cpu_state_dict(module: nn.Module) -> dict:
    return {
        name: value.detach().cpu().clone()
        for name, value in module.state_dict().items()
    }


def _atomic_torch_save(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def _load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:  # Compatibility with older PyTorch versions.
        return torch.load(path, map_location=device)


def save_training_checkpoint(
    next_epoch: int,
    best_validation_top1: float,
    best_model_state_dict: dict,
    no_improvement_epochs: int,
    target_streak: int,
) -> None:
    if best_model_state_dict is None:
        best_model_state_dict = _cpu_state_dict(model)

    checkpoint = {
        "format_version": 3,
        "epoch": next_epoch,
        "best_validation_top1": best_validation_top1,
        "model_state_dict": model.state_dict(),
        "best_model_state_dict": best_model_state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "no_improvement_epochs": no_improvement_epochs,
        "target_streak": target_streak,
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }

    if torch.cuda.is_available():
        checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    _atomic_torch_save(checkpoint, CHECKPOINT_PATH)


def load_training_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return 0, -1.0, None, 0, 0

    # The full checkpoint also stores Python/NumPy RNG state.
    checkpoint = _load_torch_checkpoint(CHECKPOINT_PATH)

    # Backward compatibility: older versions saved only a model state_dict.
    if "model_state_dict" not in checkpoint:
        model.load_state_dict(checkpoint, strict=True)
        print(f"Loaded legacy weights-only checkpoint from {CHECKPOINT_PATH}; starting a new optimizer schedule.")
        return 0, -1.0, _cpu_state_dict(model), 0, 0

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint.get("scaler_state_dict", {}))

    # Ensure optimizer state tensors follow the model device after loading.
    for state in optimizer.state.values():
        for name, value in state.items():
            if torch.is_tensor(value):
                state[name] = value.to(device)

    if "rng_state" in checkpoint:
        torch.set_rng_state(checkpoint["rng_state"].cpu())
    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])

    best_model_state_dict = checkpoint.get("best_model_state_dict")
    if best_model_state_dict is None:
        best_model_state_dict = _cpu_state_dict(model)

    start_epoch = int(checkpoint.get("epoch", 0))
    best_validation_top1 = float(checkpoint.get("best_validation_top1", -1.0))
    no_improvement_epochs = int(checkpoint.get("no_improvement_epochs", 0))
    target_streak = int(checkpoint.get("target_streak", 0))
    print(
        f"Resumed training from {CHECKPOINT_PATH} at epoch {start_epoch}/{NUM_EPOCHS}; "
        f"best validation Top-1: {best_validation_top1:.3f}%"
    )
    return start_epoch, best_validation_top1, best_model_state_dict, no_improvement_epochs, target_streak


start_epoch, best_validation_top1, best_model_state_dict, no_improvement_epochs, target_streak = load_training_checkpoint()

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        use_moex = torch.rand(1).item() < MOEX_PROBABILITY

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            if use_moex:
                swap_index = torch.randperm(
                    images.size(0),
                    device=images.device,
                )
                swapped_labels = labels[swap_index]

                logits = model(
                    images,
                    swap_index=swap_index,
                    moex_norm="pono",
                    moex_epsilon=1e-5,
                    moex_layer="stem",
                    moex_positive_only=False,
                )

                batch_loss = (
                    MOEX_LAMBDA * criterion(logits, labels)
                    + (1.0 - MOEX_LAMBDA)
                    * criterion(logits, swapped_labels)
                )
            else:
                logits = model(images)
                batch_loss = criterion(logits, labels)

            batch_loss = batch_loss / ACCUMULATION_STEPS

        scaler.scale(batch_loss).backward()

        if step % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)


    validation_top1 = calculate_top1(
        model,
        validation_loader,
    )

    previous_best = best_validation_top1
    if validation_top1 > best_validation_top1:
        best_validation_top1 = validation_top1
        best_model_state_dict = _cpu_state_dict(model)

    if validation_top1 > previous_best + EARLY_STOP_MIN_DELTA:
        no_improvement_epochs = 0
    else:
        no_improvement_epochs += 1

    if validation_top1 > TARGET_VALIDATION_TOP1:
        target_streak += 1
    else:
        target_streak = 0

    scheduler.step()
    save_training_checkpoint(
        next_epoch=epoch + 1,
        best_validation_top1=best_validation_top1,
        best_model_state_dict=best_model_state_dict,
        no_improvement_epochs=no_improvement_epochs,
        target_streak=target_streak,
    )

    print(
        f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | "
        f"validation Top-1: {validation_top1:.3f}% | "
        f"best: {best_validation_top1:.3f}% | "
        f"plateau: {no_improvement_epochs}/{EARLY_STOP_PATIENCE}"
    )

    reached_target = (
        epoch + 1 >= MIN_EPOCHS
        and target_streak >= TARGET_STABLE_EPOCHS
    )
    reached_plateau = (
        epoch + 1 >= MIN_EPOCHS
        and no_improvement_epochs >= EARLY_STOP_PATIENCE
    )

    if reached_target:
        print(
            f"Early stopping: validation Top-1 stayed above "
            f"{TARGET_VALIDATION_TOP1:.2f}% for {TARGET_STABLE_EPOCHS} epochs."
        )
        break

    if reached_plateau:
        print(
            f"Early stopping: no meaningful validation improvement for "
            f"{EARLY_STOP_PATIENCE} epochs."
        )
        break

print(f"Best validation Top-1: {best_validation_top1:.3f}%")


## Final clean Top-1 test and plug-and-play checkpoint verification


In [ ]:
verification_model = LNL_MoEx_S(
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(device)

checkpoint = _load_torch_checkpoint(CHECKPOINT_PATH)
state_dict = checkpoint.get("best_model_state_dict")
if state_dict is None:
    state_dict = checkpoint.get("model_state_dict", checkpoint)
verification_model.load_state_dict(state_dict, strict=True)

test_top1 = calculate_top1(
    verification_model,
    test_loader,
)

print(f"Top-1 accuracy: {test_top1:.3f}%")
if test_top1 > TARGET_VALIDATION_TOP1:
    print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: achieved.")
else:
    print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: not achieved yet.")


In [ ]:
try:
    from google.colab import files
    files.download(str(CHECKPOINT_PATH))
except ImportError:
    print(f"Checkpoint saved at: {CHECKPOINT_PATH.resolve()}")


In [ ]:
# FLOPs and parameter reporting are intentionally omitted.
